# สมุดงานสำหรับการทดสอบและทำนายผลแบบจำลอง (Inference & Evaluation)
### โครงงานการรู้จำตัวอักษรและตัวเลขภาษาไทย 72 คลาส | วิชา Deep Learning
---
สมุดงานนี้รองรับการทดสอบใน 3 รูปแบบหลักสำหรับวันแข่งขันจริง:
1. **การทำนายภาพเดี่ยว (Single Image Inference)**: สำหรับทดสอบภาพที่อาจารย์กำหนด
2. **การทำนายภาพทั้งโฟลเดอร์ (Batch / Directory Inference)**: สำหรับรับโฟลเดอร์ภาพชุดทดสอบของอาจารย์ และส่งออกผลลัพธ์เป็นตาราง CSV
3. **การแสดงผลตารางตัวอย่าง (Visual Grid Demo)**: สุ่มภาพขึ้นมาพลอตตารางแสดงความถูกต้องและค่าความมั่นใจ


## 1. การโหลดแบบจำลองและไฟล์ Mapping ตัวอักษรไทย


In [ ]:
import os
import glob
import json
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torchvision import transforms
import torchvision.transforms.functional as TF
from torchvision.models import resnet50

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# โหลดรายชื่อคลาสและ Mapping ภาษาไทย
with open('classes.json', 'r', encoding='utf-8') as f:
    classes = json.load(f)

with open('char_mapping.json', 'r', encoding='utf-8') as f:
    char_map = json.load(f)

class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for i, c in enumerate(classes)}

# โหลดโครงสร้างโมเดลและค่าน้ำหนักที่ผ่านการฝึกสอน (best_model.pt หรือ model.pt)
model_path = 'best_model.pt' if os.path.exists('best_model.pt') else 'model.pt'

model = resnet50(weights=None)
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(classes)))
checkpoint = torch.load(model_path, map_location=device)

if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    val_acc = checkpoint.get('val_top1_acc', 'N/A')
    print(f"Loaded checkpoint {model_path} (Trained Best Val Top-1: {val_acc:.2f}%)")
else:
    model.load_state_dict(checkpoint)
    print(f"Loaded weights {model_path}")

model = model.to(device)
model.eval()

# กำหนด Transformation สำหรับภาพนำเข้า
img_size = 224
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print(f"System ready for 72 classes inference!")


## 2. การทำนายภาพเดี่ยว (Single Image Inference)
ฟังก์ชัน `predict_image()` รองรับการทำนายทีละภาพ พร้อมแสดงผลลัพธ์ **Top-K (เช่น Top-3)**, ตัวอักษรไทย, คำอธิบาย และค่าความมั่นใจ (Confidence Score)


In [ ]:
def predict_image(img_path, topk=3, use_tta=False, show_image=True):
    if not os.path.exists(img_path):
        print(f"Error: File not found {img_path}")
        return None
        
    image = Image.open(img_path).convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        if use_tta:
            # Test-Time Augmentation (TTA): หมุนซ้าย/ขวาเล็กน้อยและเฉลี่ยผล
            angles = [-5, 0, 5]
            probs_list = []
            for ang in angles:
                rot = TF.rotate(tensor, ang, fill=1.0)
                out = model(rot)
                probs_list.append(torch.softmax(out, dim=1))
            probs = torch.stack(probs_list).mean(dim=0).squeeze(0)
        else:
            outputs = model(tensor)
            probs = torch.softmax(outputs, dim=1).squeeze(0)
            
    top_probs, top_indices = torch.topk(probs, min(topk, len(classes)))
    
    results = []
    print(f"=== ผลการทำนาย: {os.path.basename(img_path)} ===")
    for rank in range(min(topk, len(classes))):
        cls_code = idx_to_class[top_indices[rank].item()]
        prob_pct = top_probs[rank].item() * 100.0
        info = char_map.get(str(cls_code), {})
        char_txt = info.get('char', 'N/A')
        desc = info.get('description', '')
        
        results.append({
            'rank': rank + 1,
            'class_code': cls_code,
            'char': char_txt,
            'description': desc,
            'confidence': prob_pct
        })
        marker = ">>>" if rank == 0 else "   "
        print(f"{marker} อันดับ {rank + 1}: คลาส {cls_code:>3} | อักษร: {char_txt} ({desc}) | ความมั่นใจ: {prob_pct:.2f}%")
        
    if show_image:
        plt.figure(figsize=(4, 4))
        plt.imshow(image)
        plt.axis('off')
        top1 = results[0]
        plt.title(f"ทำนาย: {top1['char']} ({top1['description']})\nมั่นใจ: {top1['confidence']:.2f}%", fontsize=12, fontweight='bold')
        plt.show()
        
    return results

# ทดสอบรันภาพตัวอย่างจากชุดข้อมูล
sample_imgs = glob.glob("ThaiCharacter Dataset/round2/*/*.jpg")
if sample_imgs:
    predict_image(sample_imgs[0], topk=3)
else:
    print("ไม่พบไฟล์ตัวอย่างในชุดข้อมูล")


## 3. การทำนายทั้งโฟลเดอร์ภาพทดสอบของอาจารย์ (Batch / Directory Inference)
สำหรับการแข่งขันจริงในห้องเรียน เมื่ออาจารย์ให้โฟลเดอร์ชุดทดสอบมา (เช่น โฟลเดอร์ `test_set/`)
ฟังก์ชันนี้จะ:
- วนลูปอ่านทุกไฟล์ภาพในโฟลเดอร์นั้นอย่างรวดเร็ว
- ส่งออกผลการทำนายทั้งหมดเป็นไฟล์ตาราง **`inference_results.csv`**
- หากในโฟลเดอร์มีเฉลย (Ground Truth) โปรแกรมจะคำนวณและแสดงค่า **Top-1 และ Top-3 Accuracy (%)** ขึ้นหน้าจอทันที


In [ ]:
def predict_directory(dir_path, output_csv="inference_results.csv", use_tta=True):
    if not os.path.exists(dir_path):
        print(f"Error: Directory not found {dir_path}")
        return
        
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    img_files = []
    for root, _, files in os.walk(dir_path):
        for fn in files:
            if fn.lower().endswith(valid_exts):
                img_files.append(os.path.join(root, fn))
                
    if not img_files:
        print(f"ไม่พบไฟล์รูปภาพในโฟลเดอร์: {dir_path}")
        return
        
    print(f"พบภาพทั้งหมด {len(img_files)} ภาพใน {dir_path} กำลังเริ่มทำนาย (TTA={use_tta})...")
    
    results = []
    top1_correct, top3_correct, total_labeled = 0, 0, 0
    has_ground_truth = False
    
    for idx, fp in enumerate(img_files):
        # ทำนายผลแบบไม่แสดงภาพทีละรูปเพื่อความรวดเร็ว
        preds = predict_image(fp, topk=3, use_tta=use_tta, show_image=False)
        top1 = preds[0]
        
        # ตรวจสอบ Ground Truth จากชื่อโฟลเดอร์พ่อแม่
        parent = os.path.basename(os.path.dirname(fp))
        true_code = parent if parent in classes else None
        
        row = {
            'filename': os.path.basename(fp),
            'filepath': fp,
            'predicted_code': top1['class_code'],
            'predicted_char': top1['char'],
            'description': top1['description'],
            'confidence_pct': f"{top1['confidence']:.2f}%",
            'top2_char': preds[1]['char'] if len(preds) > 1 else '',
            'top3_char': preds[2]['char'] if len(preds) > 2 else ''
        }
        
        if true_code:
            has_ground_truth = True
            is_correct = (top1['class_code'] == true_code)
            is_top3 = any(p['class_code'] == true_code for p in preds)
            row['ground_truth'] = true_code
            row['correct'] = is_correct
            if is_correct: top1_correct += 1
            if is_top3: top3_correct += 1
            total_labeled += 1
            
        results.append(row)
        if (idx + 1) % 50 == 0 or (idx + 1) == len(img_files):
            print(f"ประมวลผลแล้ว [{idx + 1}/{len(img_files)}] ภาพ...")
            
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n[สำเร็จ] บันทึกผลการทำนายทั้งหมดลงในไฟล์: {output_csv}")
    
    if has_ground_truth and total_labeled > 0:
        print("=" * 50)
        print(f"ผลการประเมินความแม่นยำบน {total_labeled} ภาพ:")
        print(f"  --> Top-1 Accuracy: {(top1_correct / total_labeled) * 100.0:.2f}%")
        print(f"  --> Top-3 Accuracy: {(top3_correct / total_labeled) * 100.0:.2f}%")
        print("=" * 50)
        
    return df_results

# คำสั่งสำหรับรันชุดทดสอบของอาจารย์ (เปลี่ยน path เป็นโฟลเดอร์จริงของอาจารย์ในวันแข่ง):
# predict_directory('path_to_teacher_test_folder/')


## 4. การแสดงผลตารางตัวอย่างผลการทำนาย (Visual Grid Demo)
สุ่มภาพตัวอย่างขึ้นมา 12 ภาพ เพื่อแสดงตารางเปรียบเทียบระหว่างตัวอักษรจริงกับตัวอักษรที่แบบจำลองทำนาย
- **สีเขียว**: ทำนายถูกต้อง
- **สีแดง**: ทำนายผิดพลาด


In [ ]:
def show_visual_grid(num_samples=12):
    all_imgs = glob.glob("ThaiCharacter Dataset/round2/*/*.jpg")
    if not all_imgs:
        print("ไม่พบภาพในชุดข้อมูล")
        return
        
    sampled = np.random.choice(all_imgs, min(num_samples, len(all_imgs)), replace=False)
    
    cols = 4
    rows = (len(sampled) + cols - 1) // cols
    plt.figure(figsize=(16, rows * 3.5))
    
    for i, fp in enumerate(sampled):
        true_code = os.path.basename(os.path.dirname(fp))
        true_info = char_map.get(str(true_code), {})
        true_char = true_info.get('char', true_code)
        
        preds = predict_image(fp, topk=1, use_tta=False, show_image=False)
        pred = preds[0]
        
        is_ok = (pred['class_code'] == true_code)
        color = 'darkgreen' if is_ok else 'crimson'
        status = "✓ ถูกต้อง" if is_ok else "✗ ผิด"
        
        img = Image.open(fp).convert('RGB')
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"จริง: {true_char} ({true_code})\nทาย: {pred['char']} [{pred['confidence']:.1f}%]\n{status}",
                  fontsize=11, color=color, fontweight='bold')
                  
    plt.tight_layout()
    plt.show()

# แสดงตารางตัวอย่าง
show_visual_grid(num_samples=8)
